In [17]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

In [18]:
_ROOT_DIR = "./../../"

In [19]:
def call_data():
    ret_train_df = pd.read_csv(_ROOT_DIR+ './data/raw/cell2celltrain.csv')
    ret_test_df = pd.read_csv(_ROOT_DIR+ './data/raw/cell2cellholdout.csv')


    return ret_train_df, ret_test_df

In [20]:
# Churn (Yes/No) -> 1/0
# test_df에는 churn이 NaN로 되어있음, 수정 불필요
def preprocess_churn(train_df):
    train_df['Churn'] = train_df['Churn'].map({'Yes': 1, 'No': 0})

    return train_df

In [21]:
# 그 외 인코딩
def preprocess_encoding(train_df, test_df, scaler):
    # HandsetPrice의 Unknown -> NaN으로 변환 후 중앙값으로 대체
    train_df['HandsetPrice'] = pd.to_numeric(train_df['HandsetPrice'].replace('Unknown', np.nan))
    test_df['HandsetPrice'] = pd.to_numeric(test_df['HandsetPrice'].replace('Unknown', np.nan))

    median_price = train_df['HandsetPrice'].median()
    train_df['HandsetPrice'] = train_df['HandsetPrice'].fillna(median_price)
    test_df['HandsetPrice'] = test_df['HandsetPrice'].fillna(median_price)


    # 스케일링
    numeric_cols = train_df.select_dtypes(include=['number']).columns.tolist()
    if 'CustomerID' in numeric_cols:
        numeric_cols.remove('CustomerID')

    if 'Churn' in numeric_cols:
        numeric_cols.remove('Churn')

    
    train_df[numeric_cols] = scaler.fit_transform(train_df[numeric_cols])
    test_df[numeric_cols] = scaler.transform(test_df[numeric_cols])



    # MaritalStatus의 Unknown -> NaN으로 변환 후 one-hot 인코딩
    train_df['MaritalStatus'] = train_df['MaritalStatus'].replace('Unknown', np.nan)
    test_df['MaritalStatus'] = test_df['MaritalStatus'].replace('Unknown', np.nan)
    train_df = pd.get_dummies(train_df, columns=['MaritalStatus'], prefix='Marital', dtype=int)
    test_df = pd.get_dummies(test_df, columns=['MaritalStatus'], prefix='Marital', dtype=int)


    # Yes/No -> 1/0
    binary_cols = [
    'ChildrenInHH', 'HandsetRefurbished', 'HandsetWebCapable', 'TruckOwner', 
    'RVOwner', 'BuysViaMailOrder', 'RespondsToMailOffers', 'OptOutMailings', 
    'NonUSTravel', 'OwnsComputer', 'HasCreditCard', 'NewCellphoneUser', 
    'NotNewCellphoneUser', 'OwnsMotorcycle', 'MadeCallToRetentionTeam'
    ]

    for col in binary_cols:
        train_df[col] = train_df[col].map({'Yes': 1, 'No': 0})
        test_df[col] = test_df[col].map({'Yes': 1, 'No': 0})

    
    # CreditRating
    credit_mapping = {
    '1-Highest': 1, '2-High': 2, '3-Good': 3, '4-Medium': 4, 
    '5-Low': 5, '6-VeryLow': 6, '7-Lowest': 7
    }
    train_df['CreditRating'] = train_df['CreditRating'].map(credit_mapping)
    test_df['CreditRating'] = test_df['CreditRating'].map(credit_mapping)

    # one-hot
    train_df = pd.get_dummies(train_df, columns=['PrizmCode', 'Occupation'], prefix=['Prizm', 'Occ'], dtype=int)
    test_df = pd.get_dummies(test_df, columns=['PrizmCode', 'Occupation'], prefix=['Prizm', 'Occ'], dtype=int)

    


    return train_df, test_df, scaler
    

In [22]:
# CustomerID 제거
def preprocess_customerID(train_df, test_df):
    train_df = train_df.drop(columns=['CustomerID'])
    test_df = test_df.drop(columns=['CustomerID'])
    
    return train_df, test_df

In [23]:
# Service area 제거
# 비슷한 값을 가지고 있는 PrizmCode가 존재
def preprocess_service_area(train_df, test_df):
    train_df = train_df.drop(columns=['ServiceArea'])
    test_df = test_df.drop(columns=['ServiceArea'])

    return train_df, test_df

In [24]:
# Homeownership 제거
# 필요없음
def preprocess_home_ownership(train_df, test_df):
    train_df = train_df.drop(columns=['Homeownership'])
    test_df = test_df.drop(columns=['Homeownership'])

    return train_df, test_df

In [25]:
def base_preprocess(train_df, test_df, scaler):
    train_df = preprocess_churn(train_df)
    train_df, test_df = preprocess_customerID(train_df, test_df)
    train_df, test_df, scaler = preprocess_encoding(train_df, test_df, scaler)
    train_df, test_df = preprocess_service_area(train_df, test_df)
    train_df, test_df = preprocess_home_ownership(train_df, test_df)

    return train_df, test_df, scaler

In [26]:
def remove_null_rows(train_df, test_df, columns):
    ret_train_df = train_df.dropna(subset=columns)
    ret_test_df = test_df.dropna(subset=columns)
    return ret_train_df, ret_test_df

In [27]:
def remove_missing_values(train_df, test_df):
    
    cols = [['Handsets', 'HandsetModels', 'CurrentEquipmentDays'],
    ['PercChangeRevenues', 'PercChangeMinutes'],
    ['DirectorAssistedCalls', 'TotalRecurringCharge', 'RoamingCalls', 'OverageMinutes', 'MonthlyRevenue', 'MonthlyMinutes'],
    ['AgeHH1', 'AgeHH2']]

    for col in cols:
        train_df, test_df = remove_null_rows(train_df, test_df, col)

    return train_df, test_df

In [28]:
# 이후 모델 학습을 용이하게 하기 위해 'Churn'열을 제거하고 따로 저장 (모델 학습 시 X, y로 나누기 위해)
def preprocess_remove_churn(train_df, test_df):
    train_churn = train_df['Churn']
    train_df = train_df.drop(columns=['Churn'])
    test_df = test_df.drop(columns=['Churn'])

    return train_df, test_df, train_churn

In [29]:
def preprocess_final(scaler):
    train_df, test_df = call_data()
    train_df, test_df, scaler = base_preprocess(train_df, test_df, scaler)
    train_df, test_df = remove_missing_values(train_df, test_df)
    train_df, test_df, train_churn = preprocess_remove_churn(train_df, test_df)

    return train_df, test_df, train_churn

In [30]:
scaler = StandardScaler()
train_df, test_df, train_churn = preprocess_final(scaler)


In [31]:
display(train_df.head())
display(test_df.head())
display(train_churn.head())

,MonthlyRevenue,MonthlyMinutes,TotalRecurringCharge,DirectorAssistedCalls,OverageMinutes,RoamingCalls,PercChangeMinutes,PercChangeRevenues,DroppedCalls,BlockedCalls,...,Prizm_Suburban,Prizm_Town,Occ_Clerical,Occ_Crafts,Occ_Homemaker,Occ_Other,Occ_Professional,Occ_Retired,Occ_Self,Occ_Student
0,-0.782676,-0.578738,-1.041153,-0.289532,-0.414422,-0.125914,-0.564836,-0.449987,-0.587303,-0.309284,...,1,0,0,0,0,0,1,0,0,0
1,-0.940180,-0.973177,-1.250809,-0.401714,-0.414422,-0.125914,0.029311,0.030120,-0.631532,-0.373230,...,1,0,0,0,0,0,1,0,0,0
2,-0.468118,-0.976952,-0.370255,-0.401714,-0.414422,-0.125914,0.037077,0.030120,-0.664703,-0.373230,...,0,1,0,1,0,0,0,0,0,0
3,0.526784,1.484048,1.181196,0.154708,-0.414422,-0.125914,0.654524,0.234797,5.085049,0.330172,...,0,0,0,0,0,1,0,0,0,0
4,-0.936810,-0.992050,-1.250809,-0.401714,-0.414422,-0.125914,0.044844,0.025066,-0.664703,-0.373230,...,0,0,0,0,0,0,1,0,0,0


,MonthlyRevenue,MonthlyMinutes,TotalRecurringCharge,DirectorAssistedCalls,OverageMinutes,RoamingCalls,PercChangeMinutes,PercChangeRevenues,DroppedCalls,BlockedCalls,...,Prizm_Suburban,Prizm_Town,Occ_Clerical,Occ_Crafts,Occ_Homemaker,Occ_Other,Occ_Professional,Occ_Retired,Occ_Self,Occ_Student
0,-0.030209,-0.080499,-0.412187,-0.289532,-0.176295,-0.125914,2.110765,1.318828,0.253046,-0.281879,...,0,0,0,0,0,1,0,0,0,0
1,-0.080987,0.083694,1.055403,-0.401714,-0.414422,-0.125914,0.192410,0.030120,0.407847,-0.309284,...,0,0,0,0,0,0,1,0,0,0
2,0.865158,0.968824,0.132918,1.819487,3.933984,-0.125914,0.813740,0.618883,0.739563,-0.099177,...,1,0,0,1,0,0,0,0,0,0
3,-0.522267,-0.703298,-0.705704,-0.401714,-0.248768,-0.125914,0.161343,0.214582,-0.443559,-0.309284,...,0,0,0,0,0,1,0,0,0,0
4,-0.080089,1.297209,0.132918,-0.069656,-0.414422,0.006494,0.701124,0.055389,-0.366159,0.174876,...,1,0,0,0,0,1,0,0,0,0


0    1
1    1
2    0
3    0
4    1
Name: Churn, dtype: int64

In [32]:
train_df.to_csv(_ROOT_DIR + './data/preprocessed/cell2cell_train.csv', index=False)
test_df.to_csv(_ROOT_DIR + './data/preprocessed/cell2cell_holdout.csv', index=False)    # test용이 아니라 시연용? 이었음
train_churn.to_csv(_ROOT_DIR + './data/preprocessed/cell2cell_train_churn.csv', index=False)